# 设施选址问题 (FLP)

**类别:** 选址

来源: [https://www.hexaly.com/templates/facility-location-problem-flp](https://www.hexaly.com/templates/facility-location-problem-flp)


## 问题描述

**设施选址问题 (FLP)** 的定义如下。给定一组地点以及每对地点之间的运输成本,从中选取 p 个地点作为设施,以最小化运输成本。一个地点的运输成本等于其到最近设施的距离。因此,目标是为设施提供最优布局以最小化运输成本。该问题也称为 P-Median 问题。

	

### 学到的要点

- 添加 [布尔决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#boolean-decisions) 来建模一个地点是否被选为设施
- 使用 [非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算每个地点的运输成本
- 了解 Hexaly Optimizer 的建模风格:[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 数据

我们提供的数据文件来自 [OR-LIB](http://people.brunel.ac.uk/~mastjjb/jeb/orlib/pmedinfo.html)。其格式如下:

- 地点数量
- 原始实例中的边数
- 要选择的设施数量
- 距离矩阵,使用 Floyd 算法从原始文件计算得出。


## 模型

设施选址问题 (FLP) 的 Hexaly 模型使用布尔决策变量来表示每个地点是否被选为设施。我们将这些布尔值的和约束为不超过 p,以确保设施数量最多为 p。

然后我们需要计算运输成本。注意,在模型中无需为每个地点定义其最近设施,只需计算该地点与其最近设施之间的距离即可。利用三元算子,我们可以计算地点 i 与设施 j 之间的运输成本:若 j 为设施则等于 i 与 j 的距离,否则为无穷大。利用对所有可能设施的 **min** 算子,我们可以计算地点 i 与其最近设施之间的距离,对所有 i 均如此。

最后,需要最小化的目标即为这些运输成本之和。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

#
# Read instance data
#
def read_instance(instance_file):
    file_it = iter(read_integers(instance_file))
    # Number of locations
    N = next(file_it)
    next(file_it) # Skip number of edges
    # Size of the subset S of facilities
    p = next(file_it)

    # w: Weight matrix of the shortest path between locations
    # wmax: Maximum distance between two locations
    wmax = 0
    w = [None] * N
    for i in range(N):
        w[i] = [None] * N
        for j in range(N):
            w[i][j] = next(file_it)
            if w[i][j] > wmax:
                wmax = w[i][j]

    return N, p, wmax, w

def main(instance_file, output_file, time_limit):
    N, p, wmax, w = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        m = optimizer.model

        # One variable for each location: 1 if facility, 0 otherwise
        x = [m.bool() for i in range(N)]

        # No more than p locations are selected to be facilities
        opened_locations = m.sum(x[i] for i in range(N))
        m.constraint(opened_locations <= p)

        # Costs between location i and j is w[i][j] if j is a facility or 2 * wmax if not
        costs = [None] * N
        for i in range(N):
            costs[i] = [None] * N
            for j in range(N):
                costs[i][j] = m.iif(x[j], w[i][j], 2 * wmax)

        # Cost between location i and the closest facility
        cost = [None] * N
        for i in range(N):
            cost[i] = m.min(costs[i][j] for j in range(N))

        # Minimize the total cost
        total_cost = m.sum(cost[i] for i in range(N))
        m.minimize(total_cost)

        m.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - value of the objective
        # - indices of the facilities (between 0 and N-1)
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d\n" % total_cost.value)
                for i in range(N):
                    if x[i].value == 1:
                        f.write("%d " % i)
                f.write("\n")

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python facility_location.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20
    main(instance_file, output_file, time_limit)
